# Adaptive Learning and Predictive Intelligence Platform
## Exploratory Data Analysis (EDA) on Open University Learning Analytics Dataset (OULAD)

**Author:** Machine Learning Intern, MintX Solutions  
**Project:** Adaptive Learning & Predictive Intelligence Platform  
**Historical Internship Period:** 15 June 2026 – 14 July 2026  
**Dataset:** Open University Learning Analytics Dataset (OULAD)  
**DOI:** [10.1038/sdata.2017.171](https://doi.org/10.1038/sdata.2017.171)

---
### Objectives
1. Ingest and profile core OULAD relational tables (`studentInfo`, `studentRegistration`, `studentAssessment`, `assessments`, `studentVle`, `vle`, `courses`).
2. Examine missing values, data types, and demographic distributions.
3. Investigate Virtual Learning Environment (VLE) interaction patterns prior to the 100-day early-warning cutoff.
4. Analyze assessment score trajectories and submission latencies.
5. Validate leakage prevention protocols and group-based data partitioning.

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

print("Raw Data Path:", DATA_DIR.resolve())
print("Processed Data Path:", PROCESSED_DIR.resolve())

## 1. Data Ingestion & Table Schemas

In [2]:
df_student_info = pd.read_csv(DATA_DIR / "studentInfo.csv")
df_student_reg = pd.read_csv(DATA_DIR / "studentRegistration.csv")
df_student_assess = pd.read_csv(DATA_DIR / "studentAssessment.csv")
df_assessments = pd.read_csv(DATA_DIR / "assessments.csv")
df_student_vle = pd.read_csv(DATA_DIR / "studentVle.csv")
df_vle = pd.read_csv(DATA_DIR / "vle.csv")
df_courses = pd.read_csv(DATA_DIR / "courses.csv")

print(f"Students: {len(df_student_info):,} records, {df_student_info['id_student'].nunique():,} unique students")
print(f"Registrations: {len(df_student_reg):,} records")
print(f"Assessment Submissions: {len(df_student_assess):,} records")
print(f"VLE Interactions: {len(df_student_vle):,} records across {df_vle['id_site'].nunique():,} learning sites")
print(f"Courses: {len(df_courses)} presentations")

## 2. Demographic and Academic Outcome Distribution

In [3]:
outcome_counts = df_student_info["final_result"].value_counts()
outcome_pcts = df_student_info["final_result"].value_counts(normalize=True) * 100

outcome_df = pd.DataFrame({"Count": outcome_counts, "Percentage (%)": outcome_pcts.round(2)})
print("Academic Outcome Distribution:")
print(outcome_df)

# Define early academic risk: Fail or Withdrawn
at_risk_mask = df_student_info["final_result"].isin(["Fail", "Withdrawn"])
print(f"\nOverall At-Risk Proportion: {at_risk_mask.mean():.2%}")

## 3. Early VLE Engagement Dynamics (Day <= 100)
To enable proactive academic interventions without lookahead bias, features are strictly calculated up to Day 100.

In [4]:
CUTOFF_DAY = 100
vle_early = df_student_vle[df_student_vle["date"] <= CUTOFF_DAY]

early_student_engagement = vle_early.groupby("id_student").agg(
    total_clicks=("sum_click", "sum"),
    active_days=("date", "nunique"),
    distinct_resources=("id_site", "nunique")
).reset_index()

early_student_engagement["clicks_per_active_day"] = (
    early_student_engagement["total_clicks"] / early_student_engagement["active_days"]
)

print("Early VLE Engagement Summary Statistics:")
print(early_student_engagement.describe().round(2))

## 4. Assessment Performance & Submission Latency

In [5]:
assess_merged = df_student_assess.merge(
    df_assessments[["id_assessment", "code_module", "code_presentation", "date", "weight"]],
    on="id_assessment",
    how="inner"
)
assess_early = assess_merged[assess_merged["date"] <= CUTOFF_DAY].copy()
assess_early["submission_delay"] = assess_early["date_submitted"] - assess_early["date"]
assess_early["is_late"] = (assess_early["submission_delay"] > 0).astype(int)

student_assess_summary = assess_early.groupby("id_student").agg(
    assessments_taken=("id_assessment", "count"),
    mean_score=("score", "mean"),
    late_submissions=("is_late", "sum"),
    mean_delay=("submission_delay", "mean")
).reset_index()

print("Early Assessment Performance Summary:")
print(student_assess_summary.describe().round(2))

## 5. Feature Engineering and Group-Based Splitting Verification
Verify that training and test splits produced by the feature engineering pipeline share zero student identities (`GroupShuffleSplit`).

In [6]:
train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")

train_students = set(train_df["id_student"].unique())
test_students = set(test_df["id_student"].unique())

overlap = train_students.intersection(test_students)
print(f"Train samples: {len(train_df)} ({len(train_students)} unique students)")
print(f"Test samples: {len(test_df)} ({len(test_students)} unique students)")
print(f"Student identity overlap between splits: {len(overlap)} (Zero leakage confirmed!)")
print(f"Train risk proportion: {train_df['target_risk'].mean():.2%}")
print(f"Test risk proportion: {test_df['target_risk'].mean():.2%}")